# 01. 규칙 기반 손 추종 / 60분

**강사 설명 → 완성된 셀 실행 → 조건 비교 → 결과 해설** 순서입니다. 코드 작성 과제나 관찰 답안 제출은 없습니다.

손 모양은 이동 허용 여부를, 손 위치는 목표를 정합니다. 좌표가 있어도 주먹·도착·시간 만료라면 정지합니다.

[단원 실행 안내](README.md) · [슬라이드 실습 시작](https://docs.google.com/presentation/d/1J5Jw6Lg-2oYFKec-RROYlt6nxtG-IHVHd7hLwH8gd9s/edit#slide=id.d4_lab01_start)

각 단원은 새 커널에서 시작합니다. 노트북 셀은 위에서 아래로 실행하고, `bash` 명령은 지정한 터미널에서 실행합니다. 결과가 이미 있으면 재사용 여부와 실행 이름을 확인합니다.


## 1. 아키텍처 — 지금 실행하는 부분

![규칙 기반 손 추종의 입력·처리·출력](../교재그림/guided-vla/day4-01.svg)

target_xy는 유한한 정규화 좌표를 검사한 뒤 [1,10] 범위로 변환합니다. follow_motion은 먼저 시간과 판단을 검사하고, 통과한 경우에만 pose에서 목표까지의 거리와 각도 오차를 계산합니다. 회전이 필요한 동안 선속도는 0일 수 있으며, 이는 FOLLOW와 모순되지 않습니다.

**그림을 읽는 순서:** 1→2→3 뒤 아래 4→5→6으로 이어집니다. 실시간 반복은 새 관측이 다음 입력으로 들어오는 과정이며, 저장 결과를 읽는 셀은 모델을 다시 실행하지 않습니다.


### 전체 실행 지도에서 현재 단원 찾기

![4일차 실행 경계 전체 지도](../교재그림/guided-vla/detailed/day4-system.svg)

터미널 A/B/C는 실행·관찰 역할이고 ROS 노드와 HTTP 서비스는 실제 처리 주체입니다. 02는 파일 요청만, 05는 독립 YOLO 실행만 다루므로 매 단원에서 모든 상자를 동시에 실행하지 않습니다. 아래 단원별 도식과 함께 현재 경로를 확인합니다.

## 상세 해설 — 앞선 수업 없이도 흐름을 읽기

아래 설명은 1~3일차에서 배운 개념도 이번 실습의 입력·코드·출력에 맞춰 다시 풉니다. 실행 중에는 필요한 부분을 읽고, 수업 뒤에는 독립적인 참고 자료로 사용할 수 있습니다. 추가 작성 과제는 없습니다.

### 1. 기초 복습: 영상 좌표와 속도 명령

- 픽셀은 색 값, landmark는 정규화된 위치입니다.
- 손 중심은 0·5·9·13·17번 점의 평균입니다.
- 영상 y는 아래로, turtlesim y는 위로 증가합니다.
- Twist는 목표 위치가 아니라 선속도·각속도입니다.

카메라 영상은 높이×너비×채널의 숫자 배열입니다. OpenCV의 BGR과 MediaPipe가 받는 RGB는 채널 순서가 다르므로 cvtColor로 바꿉니다. 같은 세 숫자라도 순서가 다르면 다른 색으로 해석됩니다. MediaPipe가 돌려주는 21개 landmark의 x·y는 영상 크기로 정규화한 위치입니다. 화면의 픽셀 위치와 0~1 좌표를 같은 숫자로 취급하면 안 됩니다.

hand_observation은 손이 정확히 하나이고 21개 좌표가 유효할 때 손바닥 중심을 계산합니다. 손끝까지 평균내면 손가락을 펴고 접는 것만으로 목표가 크게 흔들릴 수 있으므로 손바닥을 이루는 0·5·9·13·17번을 사용합니다. 손목과 중지 뿌리의 거리가 너무 작거나 관절 간 거리가 거의 0이면 불안정한 관측으로 취급합니다. 손가락 거리비로 만든 FIST/NON_FIST 규칙은 이 수업의 휴리스틱이며 보편적인 손 동작 분류기가 아닙니다.

turtlesim의 Pose는 현재 x·y·theta이고 Twist는 앞으로 가는 속도 linear.x와 회전 속도 angular.z입니다. 손이 오른쪽에 있다는 사실을 곧바로 우회전 명령으로 바꾸지 않습니다. 먼저 목표 위치를 정한 뒤 현재 거북이의 위치와 방향을 이용해 회전과 전진을 계산합니다.


### 2. 실행 아키텍처: 세 터미널과 피드백

![실행 주체와 데이터 경계](../교재그림/guided-vla/detailed/day4-01.svg)

- A: 카메라 → /camera/image_raw → 제어 노드
- B: launch → 제어기·turtlesim·overlay 창
- 제어기 → /turtle1/cmd_vel → turtlesim
- /turtle1/pose → 제어기; C는 토픽·로그 확인

터미널은 프로그램이 아닙니다. A에서 카메라 노드를 실행하면 영상 메시지가 계속 발행되고, B에서 launch를 실행하면 제어 노드와 turtlesim 및 영상 관찰 창이 구성됩니다. 카메라는 이 launch 안에 없으므로 A의 실행이 선행되어야 합니다. C는 이미 움직이는 시스템의 토픽이나 로그를 읽는 관찰용 터미널입니다. C를 열었다고 새로운 제어기가 필요한 것은 아닙니다.

ROS 토픽은 이름과 메시지 타입이 맞는 발행자·구독자를 연결합니다. /camera/image_raw에는 Image, /turtle1/pose에는 Pose, /turtle1/cmd_vel에는 Twist가 흐릅니다. Image의 header 시각은 영상이 만들어진 시점을 식별하는 근거입니다. 손 좌표와 판정에는 같은 frame_id를 붙여 어느 이미지에서 나온 정보인지 추적합니다.

제어기는 영상을 받아 손을 인식하는 작업과 50ms마다 속도를 계산하는 작업을 분리합니다. 손 위치가 유지되어도 거북이의 Pose가 바뀌므로 목표까지의 거리와 방향 오차는 매번 달라집니다. 이것이 피드백 제어입니다. overlay와 /vla/action은 이 계산을 보여 주는 관찰 경로이며 제어 명령의 대체 입력이 아닙니다.


### 3. 코드 추적: 관측에서 회전·전진까지

- hand_observation → rule_decision → follow_motion
- 시간·판정·frame·pose 검사 후 target_xy 호출
- 거리 ≤ 0.15이면 도착; 각도 오차가 크면 회전만
- 선속도 ≤ 1.2, 각속도는 −1.5~1.5로 제한

rule_decision은 관측 state가 NON_FIST일 때 FOLLOW를 제안하고 나머지는 STOP을 제안합니다. 이 함수에는 현재 시각이나 거북이 Pose가 없으므로 실제로 지금 움직여도 되는지 판단할 수 없습니다. follow_motion이 촬영 시각, 판단 유효기간, Pose 수신 시각, 판정 형식과 frame 일치를 순서대로 확인합니다. 앞 검사가 실패하면 뒤의 목표 계산으로 넘어가지 않습니다.

target_xy([x,y])는 (1+9x, 1+9(1-y))를 반환합니다. mirror=True이면 x 대신 1-x를 사용합니다. 예를 들어 [0.2,0.3]은 (2.8,7.3), 미러링하면 (8.2,7.3)입니다. 유효 영역을 1~10으로 둬 화면 끝이 곧바로 turtlesim 경계가 되지 않게 합니다. y 반전과 x 미러링은 서로 다른 변환입니다.

목표까지 거리는 hypot으로, 방향은 atan2로 구합니다. theta와의 차이는 sin·cos·atan2로 −π~π 범위의 짧은 회전으로 정규화합니다. 방향 오차 절댓값이 0.6 이상이면 linear_x는 0이지만 angular_z는 나올 수 있습니다. FOLLOW인데 제자리에서 도는 것은 정상 분기입니다. 거리 0.15 이내는 target_reached로 정지합니다.


### 4. 출력 해석과 수정의 영향

- mirror 비교는 목표 x만 바뀌는지 확인합니다.
- STOP의 reason으로 주먹·만료·도착을 구별합니다.
- 좌표 변환 변경과 속도 상한 변경을 분리합니다.
- 단위·회전 방향·유효성 검사를 함께 유지합니다.

노트북의 고정 입력 표는 좌표와 판정 규칙만 분리해 확인합니다. 실제 카메라에서 손을 움직였을 때의 지연이나 인식 정확도는 이 표로 측정되지 않습니다. 실시간 화면에서는 손 위치, 표시된 목표, 거북이 방향, reason을 함께 보세요. 선속도가 0이라는 한 값만 보고 인식 실패라고 결론 내릴 수 없습니다.

직접 수정한다면 좌표 범위는 target_xy, 손 모양 기준은 hand_observation, 제어 게인과 도착 임계값은 follow_motion에서 바꿉니다. 세 가지를 동시에 바꾸면 출력 차이의 원인을 찾기 어렵습니다. x 미러링을 영상 전처리와 목표 변환 양쪽에 넣으면 두 번 반전되어 의도와 달라집니다. 카메라 입력의 원본 좌표계를 기준으로 어느 경계에서 변환했는지 표시해야 합니다.

명령을 바꿔 재실행할 때는 기존 B의 제어기를 종료합니다. 두 노드가 같은 cmd_vel에 발행하면 마지막으로 도착한 메시지가 행동을 덮어써 실험 조건이 섞입니다. 코드의 의도와 실행 프로세스 수가 일치해야 비교표와 실제 움직임을 연결할 수 있습니다.


## 2. 코드 위치와 변경 범위

| 구분 | 이 단원에서 확인할 내용 |
|---|---|
| 실제 소스 | [ros_ws/src/nepes_vla_demo/nepes_vla_core/hand_follow.py](ros_ws/src/nepes_vla_demo/nepes_vla_core/hand_follow.py) |
| 바꾸어 볼 조건 | `mirror:=false / true` |
| 예상되는 차이 | 같은 손 위치에서 mirror만 바꾸면 목표 x가 반대쪽으로 바뀝니다. 주먹을 쥐면 좌표가 남아도 빨간 점과 STOP이 나타납니다. |
| 직접 구현·수정할 때 | target_xy의 x 미러링과 y축 반전은 각각 한 번만 적용합니다. 카메라 입력 자체를 뒤집으면 목표 변환의 mirror와 중복될 수 있습니다. 속도 상한을 바꾸기 전에 목표 도달과 회전 분기를 다시 확인합니다. |

설정값 변경은 아래 준비된 비교 실행으로 진행합니다. 소스를 직접 변경하는 경우에는 실행 중인 프로세스를 종료하고 다시 실행해야 합니다. ROS 패키지 파일은 재빌드·환경 source까지 확인합니다.


## 3. 준비 — 실제 실행 코드와 같은 함수를 사용합니다

아래 CPU 예제는 카메라·모델·ROS 없이 자료의 흐름을 분리해 보여 줍니다. 통과가 실제 인식 성공이나 장비 성능을 뜻하지는 않습니다.


In [ ]:
from pathlib import Path
import sys, json, inspect, uuid
from datetime import datetime, timezone
from IPython.display import display, Markdown, Code
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'09.Pseudo-VLA_v2').is_dir())
LESSON = ROOT/'09.Pseudo-VLA_v2'
if str(LESSON) not in sys.path:
    sys.path.insert(0, str(LESSON))
# 매 단원 새 커널: 다른 일차의 common 모듈이 남아 있지 않게 합니다.
RUN = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8]
OUT = ROOT/'Data/outputs/09.Pseudo-VLA_v2/guided'/RUN

# 각 행은 고정 입력의 한 조건입니다. 합성 예제 통과를 실제 장비의 정확도·속도로 해석하지 않습니다.
def show_rows(rows):
    """조건·결과를 한 표에서 읽습니다. 수강생이 답안 dict를 채우지 않습니다."""
    if not rows:
        print('표시할 결과 없음')
        return
    keys = list(dict.fromkeys(key for row in rows for key in row))
    def value(v):
        return str(v).replace('|', '/').replace('\n', ' ')
    lines = [' | '.join(keys), ' | '.join(['---']*len(keys))]
    lines += [' | '.join(value(row.get(key, '')) for key in keys) for row in rows]
    display(Markdown('\n'.join(lines)))

# 비교 출력을 자동 저장합니다. 수강생이 관찰 답안을 작성하거나 수치를 옮겨 적지 않습니다.
def save_demo(name, rows):
    """합성 예제임을 파일에도 명시합니다. 모델 측정값이나 수강생 답안이 아닙니다."""
    OUT.mkdir(parents=True, exist_ok=True)
    path = OUT/(name+'.json')
    path.write_text(json.dumps({'kind':'synthetic-guided-demo', 'model_measured':False,
                               'rows':rows}, ensure_ascii=False, indent=2)+'\n')
    print('자동 저장:', path)
print('실행 폴더:', LESSON)


### 완성 코드 읽기

아래 셀은 실행에 쓰는 함수 본문을 그대로 표시합니다. **입력 검사 → 조건 분기 → 결과 반환**을 순서대로 읽습니다. 노트북에 복사한 별도 정답과 실제 동작이 어긋나지 않도록 같은 소스를 조회합니다.


In [ ]:
from common.hand_follow import target_xy, rule_decision, follow_motion
# 함수의 실제 파일과 주석을 함께 표시합니다. 이 셀은 소스를 수정하지 않습니다.
for function in (target_xy, rule_decision, follow_motion,):
    print(inspect.getsourcefile(function))
    display(Code(inspect.getsource(function), language='python'))


### 한 줄씩 읽는 처리 과정

target_xy는 유한한 정규화 좌표를 검사한 뒤 [1,10] 범위로 변환합니다. follow_motion은 먼저 시간과 판단을 검사하고, 통과한 경우에만 pose에서 목표까지의 거리와 각도 오차를 계산합니다. 회전이 필요한 동안 선속도는 0일 수 있으며, 이는 FOLLOW와 모순되지 않습니다.

target_xy의 x 미러링과 y축 반전은 각각 한 번만 적용합니다. 카메라 입력 자체를 뒤집으면 목표 변환의 mirror와 중복될 수 있습니다. 속도 상한을 바꾸기 전에 목표 도달과 회전 분기를 다시 확인합니다.


## 4. 조건을 바꾸어 실행하고 표로 비교하기

같은 손 위치에서 mirror만 바꾸면 목표 x가 반대쪽으로 바뀝니다. 주먹을 쥐면 좌표가 남아도 빨간 점과 STOP이 나타납니다.

아래 입력은 경계 조건을 설명하기 위한 고정 예제입니다. 주석을 읽고 셀 전체를 실행하면 비교표와 검사 결과가 자동으로 나옵니다.


In [ ]:
from common.hand_follow import target_xy, rule_decision, follow_motion

# 같은 입력에서 mirror 하나만 바꿉니다. y는 영상 아래쪽이 +이므로 항상 한 번 반전합니다.
rows = []
for center in ([0, 0], [0.5, 0.5], [1, 1]):
    for mirror in (False, True):
        rows.append(dict(center=center, mirror=mirror, target=target_xy(center, mirror=mirror)))
# 각 행은 고정 입력의 한 조건입니다. 합성 예제 통과를 실제 장비의 정확도·속도로 해석하지 않습니다.
show_rows(rows)
assert target_xy([0, 0]) == (1, 10)
assert target_xy([0, 0], mirror=True) == (10, 10)

# 좌표를 고정하고 손 상태만 바꿉니다. 판단과 제어가 다른 단계임을 확인합니다.
rows = []
for state in ('NON_FIST', 'FIST', 'NO_HAND'):
    observation = dict(frame_id='demo-1', captured_ms=1000,
                       center=None if state == 'NO_HAND' else [1, .5], state=state)
    decision = rule_decision(observation)
    action = follow_motion(observation, decision, (5.5, 5.5, 0),
                           now_ms=1100, issued_ms=1050, pose_ms=1100)
    rows.append(dict(state=state, judgment=decision['command'], **action))
# 각 행은 고정 입력의 한 조건입니다. 합성 예제 통과를 실제 장비의 정확도·속도로 해석하지 않습니다.
show_rows(rows)
assert [r['command'] for r in rows] == ['FOLLOW', 'STOP', 'STOP']
# 비교 출력을 자동 저장합니다. 수강생이 관찰 답안을 작성하거나 수치를 옮겨 적지 않습니다.
save_demo('coordinate-and-state', rows)


## 5. 실제 입력 실행 — 화면과 로그 연결

본수업은 로컬 카메라 화면을 함께 보며 역할을 교대합니다. 아래의 A/B/C는 터미널 역할입니다. 카메라와 제어기는 각각 하나만 실행합니다. 실제 실행의 지연과 인식 결과는 위 합성 예제와 별도로 확인합니다.


## 실제 ROS 실행 — JupyterLab 터미널 A·B·C

<!-- procedure:ros-terminals-57bc75:start -->
![새 터미널 준비와 세 역할](../교재그림/절차/ros-terminals.svg)

1–3을 새 터미널마다 적용합니다. 4–6은 동시에 실행하는 역할이며 노드·토픽 이름은 해당 실습 명령을 따릅니다.
<!-- procedure:ros-terminals-57bc75:end -->


새 Terminal에서는 아래 블록의 첫 줄 bash부터 전체를 복사합니다. 로컬·원격 모두 오프라인 수기 서명 명부의 본인 번호를 ROS_DOMAIN_ID로 사용합니다. 예시 7을 본인 번호로 바꾸고 본인 A·B·C 터미널에 같은 값을 적용합니다. 공동 통신 실습에서만 강사가 지정한 동일 명부 번호를 사용합니다. 노트북 Python 셀에 붙이지 않습니다. 이미 환경을 설정한 같은 B에서 모드·TTL만 바꿀 때는 Ctrl+C 후 launch 명령만 실행합니다.

**처음 준비 또는 패키지를 찾지 못할 때 — 빌드:**
```bash
bash
bash /workspace/08.ROS_v2/scripts/build_workspace.sh
bash /workspace/09.Pseudo-VLA_v2/scripts/build_workspace.sh
```

**A — 카메라 입력:** 이미 A가 실행 중이면 그대로 유지합니다. 카메라 publisher는 하나만 실행합니다.
```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS_v2/install/setup.bash
source /workspace/Data/build/09.Pseudo-VLA_v2/install/setup.bash
export ROS_DOMAIN_ID=7 # 7을 본인 수기 서명 명부 번호로 변경
ros2 run v4l2_camera v4l2_camera_node --ros-args -p video_device:=/dev/video0 -p image_size:="[640, 480]" -r image_raw:=/camera/image_raw
```

카메라 번호는 배정값입니다. 녹화 입력으로 비교할 때는 A를 Ctrl+C로 종료하고 같은 A에서 아래 명령을 실행합니다. 매 조건에서 같은 시작 지점으로 재생을 다시 시작합니다.
```bash
ros2 run nepes_ros_lab recorded_image_publisher --ros-args -p image_source:=/workspace/Data/videos/07.Mediapipe/person_open_palm.avi
```

**거북이의 움직임:** 한 손이 확실한 비주먹이면 손 중심을 목표로 삼습니다. 손을 영상 오른쪽/위로 옮기면 거북이 목표도 오른쪽/위로 이동합니다. 주먹·손 없음·복수 손·불확실 판단이면 정지합니다. 목표와 방향 차이가 0.6 rad 이상이면 제자리 회전 후 전진하며, 목표 거리 0.15 이하에서 멈춥니다. 손가락이 가리키는 방향을 읽거나 일정 시간 무조건 전진하는 동작이 아닙니다.

**화면 읽기:** /vla/image_overlay의 초록색 점은 전진·회전 중, 빨간색 점은 좌표가 있지만 정지 중입니다. 좌표가 없으면 NO TARGET입니다. Target은 거북이 목표 좌표, age는 제어에 사용한 프레임 경과 시간입니다. 승인 경과 시간·TTL·남은 유효시간도 확인합니다. VLM 추론 중에는 이전 관측 화면이 유지됩니다. 최신 원본은 /camera/image_raw입니다. mirror:=true는 목표 x만 반전하고 영상과 점은 원본 위치를 유지합니다.

**B — 규칙 기반 이동:** VLM은 실행하지 않습니다. 한 손·주먹·비주먹으로 기준 동작을 관찰합니다.
```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS_v2/install/setup.bash
source /workspace/Data/build/09.Pseudo-VLA_v2/install/setup.bash
export ROS_DOMAIN_ID=7 # 7을 본인 수기 서명 명부 번호로 변경
ros2 launch nepes_vla_demo hand_follow.launch.py mode:=rule \
  ttl_ms:=400 \
  log_path:=/workspace/Data/logs/09.Pseudo-VLA_v2/01-hand-rule.jsonl
```

**C — 관측·판단·행동 확인:** 새 Terminal에서 전체를 실행합니다.
```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS_v2/install/setup.bash
source /workspace/Data/build/09.Pseudo-VLA_v2/install/setup.bash
export ROS_DOMAIN_ID=7 # 7을 본인 수기 서명 명부 번호로 변경
ros2 topic echo /vla/action
```

연속 출력은 Ctrl+C로 끝낸 뒤, 같은 C에서 필요에 따라 ros2 topic echo /vla/perception 또는 ros2 topic echo /vla/decision을 실행합니다. frame_id·reason·TTL·승인 경과 시간을 영상과 비교합니다.

**종료:** A 중단 → C에서 STOP 확인 → B 종료 → C 종료. 다음 노트북으로 바로 이어갈 때는 A를 유지하고 B만 종료합니다. GUI 없는 자습은 B launch 앞에 QT_QPA_PLATFORM=offscreen, 인자에 with_image_view:=false를 사용합니다.


### 비교 실행: 목표 좌우만 변경

기준 `mode:=rule` 실행을 B에서 Ctrl+C로 종료한 뒤 같은 터미널에서 실행합니다. 카메라 A는 유지합니다.
```bash
ros2 launch nepes_vla_demo hand_follow.launch.py mode:=rule mirror:=true \
  log_path:=/workspace/Data/logs/09.Pseudo-VLA_v2/01-rule-mirror.jsonl
```
같은 손 위치에서 거북이 목표의 x만 반대쪽으로 바뀝니다. 화면 영상 자체는 뒤집지 않습니다. 관찰 뒤 B를 종료합니다.


## 6. 결과 해설과 다음 단계

손 모양은 이동 허용 여부를, 손 위치는 목표를 정합니다. 좌표가 있어도 주먹·도착·시간 만료라면 정지합니다.

같은 손 위치에서 mirror만 바꾸면 목표 x가 반대쪽으로 바뀝니다. 주먹을 쥐면 좌표가 남아도 빨간 점과 STOP이 나타납니다.

이제 위치와 제어기는 유지하고, 손 모양을 판단하는 부분만 이미지 VLM으로 바꿉니다.

비교표와 실제 화면·로그를 강사 설명과 대조합니다. 별도 답안 작성은 없으며, 자동 저장 파일과 실행 결과로 진행 상태를 확인합니다.
